
# 08 — Profile POSet Main  
## Final 5-Variable, 5-Level Main POSet

This notebook constructs the main profile-level partial order.

It runs after:

```text
07_Pre_POSet_EDA_Checks_2002_5Var.ipynb
```

## Final decisions reflected here

- Main variable set: `baseline_5_variables`
- Final ordering variables: **5**
- Main discretization: **5 ordinal levels**
- POSet temporal design: **two single pre-shock cross-sections**
  - 2007 for the Global Financial Crisis
  - 2019 for the COVID shock
- WGI/governance is not an ordering variable.
- Recovery is not an ordering variable.
- Volatility is not an ordering variable.

## POSet construction rule

Countries are first mapped into 5-level ordinal profiles using the five final structural variables.

A profile A dominates profile B if:

```text
A is at least as high as B in every dimension
and A is strictly higher than B in at least one dimension.
```

This creates a **partial order**, not a total ranking.

## Key interpretation guardrails

- Pareto/frontier profiles are not “the best countries overall”; they are profiles that are not dominated by another profile.
- Incomparability is an important result, not a failure.
- The layer number is an analytical structure, not a scalar resilience index.
- The digit-sum is a layout/helper measure only, not a ranking.

## Main outputs

- `profile_country_map.csv`
- `profile_nodes.csv`
- `profile_dominance_relations.csv`
- `profile_hasse_edges.csv`
- `profile_layer_assignments.csv`
- `profile_layer_summary.csv`
- `profile_pareto_profiles.csv`
- `profile_pareto_countries.csv`
- `profile_incomparability_summary.csv`
- `profile_poset_quality_diagnostics.csv`
- `profile_run_summary.csv`
- `report_ready_profile_poset_notes.csv`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import itertools
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

PRE_POSET_DIR = PROJECT_ROOT / "Data" / "Processed" / "Pre_POSet_EDA"
MASTER_DIR = PROJECT_ROOT / "Data" / "Processed" / "Master_Dataset"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "08_Profile_POSet_Main"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Profile_POSet_Main"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Pre-POSet folder:", PRE_POSET_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_192218
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Pre-POSet folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Pre_POSet_EDA
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Profile_POSet_Main


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

MAIN_PROFILE_FILE_CANDIDATES = [
    PRE_POSET_DIR / "main_profile_discretization_5levels.csv",
]

ANALYSIS_SAMPLE_FILE_CANDIDATES = [
    PRE_POSET_DIR / "recommended_analysis_sample_by_shock.csv",
    MASTER_DIR / "structural_snapshot_final5_complete_cases.csv",
]

def find_first_existing_path(candidates, label):
    for path in candidates:
        if path.exists():
            print(f"Using {label}: {path}")
            return path
    raise FileNotFoundError(
        f"Could not find {label}. Tried:\n"
        + "\n".join([f"- {p}" for p in candidates])
    )

MAIN_PROFILE_FILE = find_first_existing_path(
    MAIN_PROFILE_FILE_CANDIDATES,
    "main 5-level profile file",
)

ANALYSIS_SAMPLE_FILE = find_first_existing_path(
    ANALYSIS_SAMPLE_FILE_CANDIDATES,
    "analysis sample file",
)

MAIN_VARIABLE_SET_NAME = "baseline_5_variables"
MAIN_DISCRETIZATION_LEVELS = 5

FINAL_5_SCORE_COLUMNS = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
]

LEVEL_COLUMNS = [
    "debt_capacity_level_5",
    "employment_strength_level_5",
    "rd_intensity_level_5",
    "human_capital_tertiary_level_5",
    "energy_security_level_5",
]

PROFILE_CODE_COLUMN = "profile_code_5levels"
PROFILE_DIGIT_SUM_COLUMN = "profile_digit_sum_5levels"

CONCEPT_LABELS = {
    "debt_capacity_level_5": "Debt capacity",
    "employment_strength_level_5": "Employment strength",
    "rd_intensity_level_5": "R&D intensity",
    "human_capital_tertiary_level_5": "Human capital",
    "energy_security_level_5": "Energy security",
}

print("Main variable set:", MAIN_VARIABLE_SET_NAME)
print("Level columns:")
for c in LEVEL_COLUMNS:
    print("-", c)


Using main 5-level profile file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Pre_POSet_EDA\main_profile_discretization_5levels.csv
Using analysis sample file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Pre_POSet_EDA\recommended_analysis_sample_by_shock.csv
Main variable set: baseline_5_variables
Level columns:
- debt_capacity_level_5
- employment_strength_level_5
- rd_intensity_level_5
- human_capital_tertiary_level_5
- energy_security_level_5


In [3]:

# ------------------------------------------------------
# SAVE HELPER
# ------------------------------------------------------

table_inventory_rows = []

def save_table(table, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    table.to_csv(processed_path, index=False)
    table.to_csv(diagnostics_path, index=False)
    table.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(table),
        "columns": len(table.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved:", file_name)


In [4]:

# ------------------------------------------------------
# LOAD PROFILE INPUTS
# ------------------------------------------------------

profile_input = pd.read_csv(MAIN_PROFILE_FILE)
analysis_sample = pd.read_csv(ANALYSIS_SAMPLE_FILE)

for df_ in [profile_input, analysis_sample]:
    df_["Country"] = df_["Country"].astype(str).str.strip().str.upper()
    df_["shock_id"] = df_["shock_id"].astype(str)

required_profile_cols = {"Country", "shock_id", "baseline_year"} | set(FINAL_5_SCORE_COLUMNS) | set(LEVEL_COLUMNS) | {PROFILE_CODE_COLUMN}
missing_profile_cols = required_profile_cols - set(profile_input.columns)

if missing_profile_cols:
    raise ValueError(
        f"Main profile input missing required columns: {missing_profile_cols}\n"
        f"Selected file: {MAIN_PROFILE_FILE}\n"
        f"Available columns: {list(profile_input.columns)}"
    )

# Ensure numeric types.
for col in LEVEL_COLUMNS:
    profile_input[col] = pd.to_numeric(profile_input[col], errors="coerce").astype(int)

for col in FINAL_5_SCORE_COLUMNS:
    profile_input[col] = pd.to_numeric(profile_input[col], errors="coerce")

profile_input["baseline_year"] = pd.to_numeric(profile_input["baseline_year"], errors="coerce").astype(int)

if PROFILE_DIGIT_SUM_COLUMN not in profile_input.columns:
    profile_input[PROFILE_DIGIT_SUM_COLUMN] = profile_input[LEVEL_COLUMNS].sum(axis=1)

profile_input["main_variable_set"] = MAIN_VARIABLE_SET_NAME
profile_input["levels"] = MAIN_DISCRETIZATION_LEVELS
profile_input["poset_temporal_design"] = "single_pre_shock_cross_section"

print("Profile input rows:", len(profile_input))
print("Countries by shock:")
display(profile_input.groupby(["shock_id", "baseline_year"])["Country"].nunique().reset_index(name="countries"))
display(profile_input.head())


Profile input rows: 60
Countries by shock:


,shock_id,baseline_year,countries
0,2007,2007,25
1,2019,2019,35


,Country,shock_id,baseline_year,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,debt_capacity_level_3,employment_strength_level_3,rd_intensity_level_3,human_capital_tertiary_level_3,energy_security_level_3,profile_code_3levels,profile_digit_sum_3levels,levels,debt_capacity_level_4,employment_strength_level_4,rd_intensity_level_4,human_capital_tertiary_level_4,energy_security_level_4,profile_code_4levels,profile_digit_sum_4levels,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5,profile_code_5levels,profile_digit_sum_5levels,main_variable_set,poset_temporal_design
0,AUT,2007,2007,38.5303,80.6046,68.7303,33.2757,22.6201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,4,5,3,2,2-4-5-3-2,16.0000,baseline_5_variables,single_pre_shock_cross_section
1,BEL,2007,2007,16.9811,46.5777,48.5360,53.5113,9.6139,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2,4,4,1,1-2-4-4-1,12.0000,baseline_5_variables,single_pre_shock_cross_section
2,CAN,2007,2007,64.9865,63.6697,50.3901,100.0000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,4,5,5,3-3-4-5-5,20.0000,baseline_5_variables,single_pre_shock_cross_section
3,CZE,2007,2007,76.7627,74.6107,29.3904,0.4466,50.5562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,3,3,1,5,4-3-3-1-5,16.0000,baseline_5_variables,single_pre_shock_cross_section
4,DEU,2007,2007,40.6157,31.1478,68.2315,30.9850,28.5833,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,1,4,2,3,2-1-4-2-3,12.0000,baseline_5_variables,single_pre_shock_cross_section


In [5]:

# ------------------------------------------------------
# BUILD PROFILE COUNTRY MAP AND PROFILE NODES
# ------------------------------------------------------

profile_country_map = profile_input[
    [
        "Country", "shock_id", "baseline_year",
        PROFILE_CODE_COLUMN, PROFILE_DIGIT_SUM_COLUMN,
    ] + LEVEL_COLUMNS + FINAL_5_SCORE_COLUMNS
].copy()

profile_country_map = profile_country_map.rename(columns={
    PROFILE_CODE_COLUMN: "profile_code",
    PROFILE_DIGIT_SUM_COLUMN: "profile_digit_sum",
})

profile_country_map["main_variable_set"] = MAIN_VARIABLE_SET_NAME
profile_country_map["levels"] = MAIN_DISCRETIZATION_LEVELS

node_group_cols = ["shock_id", "baseline_year", "profile_code"] + LEVEL_COLUMNS

profile_nodes = (
    profile_country_map
    .groupby(node_group_cols)
    .agg(
        countries=("Country", lambda x: ", ".join(sorted(x))),
        country_count=("Country", "nunique"),
        profile_digit_sum=("profile_digit_sum", "first"),
        mean_debt_capacity_score=("debt_capacity_score_0_100", "mean"),
        mean_employment_strength_score=("employment_strength_score_0_100", "mean"),
        mean_rd_intensity_score=("rd_intensity_score_0_100", "mean"),
        mean_human_capital_tertiary_score=("human_capital_tertiary_score_0_100", "mean"),
        mean_energy_security_score=("energy_security_score_0_100", "mean"),
    )
    .reset_index()
    .sort_values(["shock_id", "profile_digit_sum", "profile_code"], ascending=[True, False, True])
)

profile_nodes["node_id"] = profile_nodes["shock_id"].astype(str) + "__" + profile_nodes["profile_code"].astype(str)
profile_nodes["main_variable_set"] = MAIN_VARIABLE_SET_NAME
profile_nodes["levels"] = MAIN_DISCRETIZATION_LEVELS

save_table(
    profile_country_map,
    "profile_country_map.csv",
    "Profile country map",
    "Country-to-profile mapping for the final 5-variable, 5-level POSet inputs.",
)

save_table(
    profile_nodes,
    "profile_nodes.csv",
    "Profile nodes",
    "Unique profile nodes with level vectors and country membership.",
)

display(profile_nodes.head())


Saved: profile_country_map.csv
Saved: profile_nodes.csv


,shock_id,baseline_year,profile_code,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5,countries,country_count,profile_digit_sum,mean_debt_capacity_score,mean_employment_strength_score,mean_rd_intensity_score,mean_human_capital_tertiary_score,mean_energy_security_score,node_id,main_variable_set,levels
19,2007,2007,4-5-5-4-5,4,5,5,4,5,DNK,1,23.0000,74.5780,100.0000,71.3466,50.0682,82.7351,2007__4-5-5-4-5,baseline_5_variables,5
23,2007,2007,5-5-2-5-5,5,5,2,5,5,EST,1,22.0000,100.0000,84.2167,21.1546,56.5490,50.2228,2007__5-5-2-5-5,baseline_5_variables,5
9,2007,2007,2-4-5-5-4,2,4,5,5,4,USA,1,20.0000,48.4466,83.6932,75.0128,77.1390,47.1942,2007__2-4-5-5-4,baseline_5_variables,5
12,2007,2007,3-3-4-5-5,3,3,4,5,5,CAN,1,20.0000,64.9865,63.6697,50.3901,100.0000,100.0000,2007__3-3-4-5-5,baseline_5_variables,5
14,2007,2007,3-5-4-4-4,3,5,4,4,4,NLD,1,20.0000,61.3704,89.8050,41.9835,49.7807,36.9664,2007__3-5-4-4-4,baseline_5_variables,5


In [6]:

# ------------------------------------------------------
# DOMINANCE AND HASSE FUNCTIONS
# ------------------------------------------------------

def dominates(levels_a, levels_b):
    a = np.array(levels_a, dtype=int)
    b = np.array(levels_b, dtype=int)
    return bool(np.all(a >= b) and np.any(a > b))


def compute_dominance_relations(nodes_for_shock):
    rows = []

    node_records = nodes_for_shock.to_dict("records")

    for a in node_records:
        levels_a = [a[col] for col in LEVEL_COLUMNS]

        for b in node_records:
            if a["profile_code"] == b["profile_code"]:
                continue

            levels_b = [b[col] for col in LEVEL_COLUMNS]

            if dominates(levels_a, levels_b):
                rows.append({
                    "shock_id": a["shock_id"],
                    "baseline_year": a["baseline_year"],
                    "dominating_profile": a["profile_code"],
                    "dominated_profile": b["profile_code"],
                    "dominating_node_id": a["node_id"],
                    "dominated_node_id": b["node_id"],
                    "dominating_digit_sum": a["profile_digit_sum"],
                    "dominated_digit_sum": b["profile_digit_sum"],
                    "digit_sum_difference": a["profile_digit_sum"] - b["profile_digit_sum"],
                })

    return pd.DataFrame(rows)


def compute_hasse_edges(dominance_relations):
    if dominance_relations.empty:
        return dominance_relations.copy()

    pairs = set(zip(dominance_relations["dominating_profile"], dominance_relations["dominated_profile"]))
    direct_rows = []

    for _, row in dominance_relations.iterrows():
        a = row["dominating_profile"]
        b = row["dominated_profile"]

        is_transitive = False

        # Edge a -> b is transitive if there exists c such that a -> c and c -> b.
        for c in set(dominance_relations["dominating_profile"]).union(set(dominance_relations["dominated_profile"])):
            if c == a or c == b:
                continue
            if (a, c) in pairs and (c, b) in pairs:
                is_transitive = True
                break

        if not is_transitive:
            direct_rows.append(row.to_dict())

    hasse_edges = pd.DataFrame(direct_rows)
    if len(hasse_edges) > 0:
        hasse_edges["edge_type"] = "hasse_cover_relation"
    return hasse_edges


def compute_layers(nodes_for_shock, dominance_relations):
    remaining = set(nodes_for_shock["profile_code"])
    layer_rows = []
    current_layer = 1

    dominance_pairs = set()
    if not dominance_relations.empty:
        dominance_pairs = set(zip(dominance_relations["dominating_profile"], dominance_relations["dominated_profile"]))

    while remaining:
        layer_profiles = []

        for profile in sorted(remaining):
            # A profile is in the current frontier layer if no remaining profile dominates it.
            dominated_by_remaining = any((other, profile) in dominance_pairs for other in remaining if other != profile)
            if not dominated_by_remaining:
                layer_profiles.append(profile)

        if not layer_profiles:
            raise RuntimeError("Layering failed: cycle or inconsistent dominance relation detected.")

        for profile in layer_profiles:
            layer_rows.append({
                "profile_code": profile,
                "layer": current_layer,
                "layer_interpretation": "Layer 1 is the undominated frontier within the remaining profiles.",
            })

        remaining -= set(layer_profiles)
        current_layer += 1

    return pd.DataFrame(layer_rows)


def compute_incomparability(nodes_for_shock, dominance_relations):
    profiles = sorted(nodes_for_shock["profile_code"].unique())

    dominance_pairs = set()
    if not dominance_relations.empty:
        dominance_pairs = set(zip(dominance_relations["dominating_profile"], dominance_relations["dominated_profile"]))

    rows = []
    total_unordered_pairs = 0
    comparable_pairs = 0
    incomparable_pairs = 0

    for a, b in itertools.combinations(profiles, 2):
        total_unordered_pairs += 1
        a_dominates_b = (a, b) in dominance_pairs
        b_dominates_a = (b, a) in dominance_pairs

        if a_dominates_b or b_dominates_a:
            comparable_pairs += 1
            relation = "comparable"
        else:
            incomparable_pairs += 1
            relation = "incomparable"

        rows.append({
            "profile_a": a,
            "profile_b": b,
            "relation": relation,
            "a_dominates_b": a_dominates_b,
            "b_dominates_a": b_dominates_a,
        })

    summary = {
        "total_unordered_profile_pairs": total_unordered_pairs,
        "comparable_profile_pairs": comparable_pairs,
        "incomparable_profile_pairs": incomparable_pairs,
        "comparability_ratio": comparable_pairs / total_unordered_pairs if total_unordered_pairs else np.nan,
        "incomparability_ratio": incomparable_pairs / total_unordered_pairs if total_unordered_pairs else np.nan,
    }

    return pd.DataFrame(rows), summary


In [7]:

# ------------------------------------------------------
# RUN PROFILE-LEVEL POSET
# ------------------------------------------------------

dominance_frames = []
hasse_frames = []
layer_frames = []
incomparability_pair_frames = []
quality_rows = []
run_summary_rows = []

for shock_id, nodes_for_shock in profile_nodes.groupby("shock_id"):
    nodes_for_shock = nodes_for_shock.copy()
    baseline_year = int(nodes_for_shock["baseline_year"].iloc[0])

    dominance = compute_dominance_relations(nodes_for_shock)
    hasse = compute_hasse_edges(dominance)
    layers = compute_layers(nodes_for_shock, dominance)
    incomparability_pairs, incomparability_summary = compute_incomparability(nodes_for_shock, dominance)

    if len(dominance) == 0:
        dominance = pd.DataFrame(columns=[
            "shock_id", "baseline_year", "dominating_profile", "dominated_profile",
            "dominating_node_id", "dominated_node_id", "dominating_digit_sum",
            "dominated_digit_sum", "digit_sum_difference",
        ])

    if len(hasse) == 0:
        hasse = pd.DataFrame(columns=list(dominance.columns) + ["edge_type"])

    layers["shock_id"] = shock_id
    layers["baseline_year"] = baseline_year

    incomparability_pairs["shock_id"] = shock_id
    incomparability_pairs["baseline_year"] = baseline_year

    dominance_frames.append(dominance)
    hasse_frames.append(hasse)
    layer_frames.append(layers)
    incomparability_pair_frames.append(incomparability_pairs)

    n_profiles = nodes_for_shock["profile_code"].nunique()
    n_countries = nodes_for_shock["country_count"].sum()
    n_layers = layers["layer"].max()

    frontier_profiles = layers[layers["layer"].eq(1)]["profile_code"].nunique()
    frontier_countries = nodes_for_shock.merge(
        layers[layers["layer"].eq(1)][["profile_code", "layer"]],
        on="profile_code",
        how="inner",
    )["country_count"].sum()

    quality_rows.append({
        "shock_id": shock_id,
        "baseline_year": baseline_year,
        "countries": int(n_countries),
        "unique_profiles": int(n_profiles),
        "profile_uniqueness_ratio": n_profiles / n_countries if n_countries else np.nan,
        "dominance_relations": len(dominance),
        "hasse_edges": len(hasse),
        "layers": int(n_layers),
        "frontier_profiles": int(frontier_profiles),
        "frontier_countries": int(frontier_countries),
        "comparability_ratio": incomparability_summary["comparability_ratio"],
        "incomparability_ratio": incomparability_summary["incomparability_ratio"],
        "main_variable_set": MAIN_VARIABLE_SET_NAME,
        "levels": MAIN_DISCRETIZATION_LEVELS,
    })

    run_summary_rows.append({
        "shock_id": shock_id,
        "baseline_year": baseline_year,
        "main_variable_set": MAIN_VARIABLE_SET_NAME,
        "variables": ",".join(FINAL_5_SCORE_COLUMNS),
        "levels": MAIN_DISCRETIZATION_LEVELS,
        "countries": int(n_countries),
        "unique_profiles": int(n_profiles),
        "dominance_relations": len(dominance),
        "hasse_edges": len(hasse),
        "layers": int(n_layers),
        "note": "Profile-level POSet. Not a scalar ranking.",
    })

profile_dominance_relations = pd.concat(dominance_frames, ignore_index=True)
profile_hasse_edges = pd.concat(hasse_frames, ignore_index=True)
profile_layer_assignments = pd.concat(layer_frames, ignore_index=True)
profile_incomparability_pairs = pd.concat(incomparability_pair_frames, ignore_index=True)
profile_poset_quality_diagnostics = pd.DataFrame(quality_rows)
profile_run_summary = pd.DataFrame(run_summary_rows)

save_table(
    profile_dominance_relations,
    "profile_dominance_relations.csv",
    "Profile dominance relations",
    "All profile-level dominance relations before transitive reduction.",
)

save_table(
    profile_hasse_edges,
    "profile_hasse_edges.csv",
    "Profile Hasse edges",
    "Direct cover relations after transitive reduction.",
)

save_table(
    profile_layer_assignments,
    "profile_layer_assignments.csv",
    "Profile layer assignments",
    "Iterative nondominated frontier layers for each shock-specific profile POSet.",
)

save_table(
    profile_incomparability_pairs,
    "profile_incomparability_pairs.csv",
    "Profile incomparability pairs",
    "All unordered profile pairs labelled comparable or incomparable.",
)

save_table(
    profile_poset_quality_diagnostics,
    "profile_poset_quality_diagnostics.csv",
    "Profile POSet quality diagnostics",
    "Quality diagnostics for the final 5-variable profile POSet.",
)

save_table(
    profile_run_summary,
    "profile_run_summary.csv",
    "Profile POSet run summary",
    "Run-level summary for the main profile POSet.",
)

display(profile_poset_quality_diagnostics)


Saved: profile_dominance_relations.csv
Saved: profile_hasse_edges.csv
Saved: profile_layer_assignments.csv
Saved: profile_incomparability_pairs.csv
Saved: profile_poset_quality_diagnostics.csv
Saved: profile_run_summary.csv


,shock_id,baseline_year,countries,unique_profiles,profile_uniqueness_ratio,dominance_relations,hasse_edges,layers,frontier_profiles,frontier_countries,comparability_ratio,incomparability_ratio,main_variable_set,levels
0,2007,2007,25,25,1.0000,96,52,5,8,8,0.3200,0.6800,baseline_5_variables,5
1,2019,2019,35,34,0.9714,129,64,4,12,13,0.2299,0.7701,baseline_5_variables,5


In [8]:

# ------------------------------------------------------
# MERGE LAYERS BACK TO PROFILES AND COUNTRIES
# ------------------------------------------------------

profile_nodes_with_layers = profile_nodes.merge(
    profile_layer_assignments[["shock_id", "baseline_year", "profile_code", "layer"]],
    on=["shock_id", "baseline_year", "profile_code"],
    how="left",
)

profile_country_map_with_layers = profile_country_map.merge(
    profile_layer_assignments[["shock_id", "baseline_year", "profile_code", "layer"]],
    on=["shock_id", "baseline_year", "profile_code"],
    how="left",
)

profile_country_map_with_layers["is_pareto_frontier"] = profile_country_map_with_layers["layer"].eq(1)
profile_nodes_with_layers["is_pareto_frontier"] = profile_nodes_with_layers["layer"].eq(1)

profile_pareto_profiles = profile_nodes_with_layers[
    profile_nodes_with_layers["is_pareto_frontier"]
].copy()

profile_pareto_countries = profile_country_map_with_layers[
    profile_country_map_with_layers["is_pareto_frontier"]
].copy()

profile_layer_summary = (
    profile_country_map_with_layers
    .groupby(["shock_id", "baseline_year", "layer"])
    .agg(
        profiles=("profile_code", "nunique"),
        countries=("Country", "nunique"),
        countries_list=("Country", lambda x: ", ".join(sorted(x))),
        min_profile_digit_sum=("profile_digit_sum", "min"),
        median_profile_digit_sum=("profile_digit_sum", "median"),
        max_profile_digit_sum=("profile_digit_sum", "max"),
    )
    .reset_index()
    .sort_values(["shock_id", "layer"])
)

profile_incomparability_summary = (
    profile_poset_quality_diagnostics[
        [
            "shock_id", "baseline_year",
            "countries", "unique_profiles",
            "comparability_ratio", "incomparability_ratio",
        ]
    ]
    .copy()
)

save_table(
    profile_nodes_with_layers,
    "profile_nodes_with_layers.csv",
    "Profile nodes with layers",
    "Profile nodes with assigned POSet layers.",
)

save_table(
    profile_country_map_with_layers,
    "profile_country_map_with_layers.csv",
    "Profile country map with layers",
    "Country-to-profile mapping with POSet layer and frontier flag.",
)

save_table(
    profile_pareto_profiles,
    "profile_pareto_profiles.csv",
    "Profile Pareto profiles",
    "Undominated frontier profiles by shock.",
)

save_table(
    profile_pareto_countries,
    "profile_pareto_countries.csv",
    "Profile Pareto countries",
    "Countries belonging to undominated frontier profiles.",
)

save_table(
    profile_layer_summary,
    "profile_layer_summary.csv",
    "Profile layer summary",
    "Summary of countries/profiles by POSet layer.",
)

save_table(
    profile_incomparability_summary,
    "profile_incomparability_summary.csv",
    "Profile incomparability summary",
    "Comparability and incomparability summary by shock.",
)

display(profile_layer_summary)
display(profile_pareto_countries[["shock_id", "baseline_year", "Country", "profile_code", "layer"]])


Saved: profile_nodes_with_layers.csv
Saved: profile_country_map_with_layers.csv
Saved: profile_pareto_profiles.csv
Saved: profile_pareto_countries.csv
Saved: profile_layer_summary.csv
Saved: profile_incomparability_summary.csv


,shock_id,baseline_year,layer,profiles,countries,countries_list,min_profile_digit_sum,median_profile_digit_sum,max_profile_digit_sum
0,2007,2007,1,8,8,"CAN, DNK, EST, FIN, GBR, LUX, SVN, USA",17.0000,19.0000,23.0000
1,2007,2007,2,7,7,"AUT, CZE, ESP, IRL, LTU, NLD, SWE",12.0000,16.0000,20.0000
2,2007,2007,3,5,5,"BEL, FRA, ITA, LVA, POL",9.0000,12.0000,14.0000
3,2007,2007,4,3,3,"DEU, HUN, SVK",9.0000,10.0000,12.0000
4,2007,2007,5,2,2,"GRC, PRT",7.0000,7.0000,7.0000
5,2019,2019,1,12,13,"AUS, CAN, CHE, CZE, DEU, DNK, EST, IRL, KOR, L...",16.0000,19.0000,21.0000
6,2019,2019,2,10,10,"AUT, BEL, FIN, GBR, HUN, LTU, NLD, NZL, ROU, SVN",13.0000,15.0000,19.0000
7,2019,2019,3,8,8,"BGR, ESP, FRA, HRV, LVA, PRT, TUR, ZAF",9.0000,11.0000,16.0000
8,2019,2019,4,4,4,"COL, GRC, ITA, SVK",7.0000,8.5000,11.0000


,shock_id,baseline_year,Country,profile_code,layer
2,2007,2007,CAN,3-3-4-5-5,1
5,2007,2007,DNK,4-5-5-4-5,1
7,2007,2007,EST,5-5-2-5-5,1
8,2007,2007,FIN,3-2-5-5-3,1
10,2007,2007,GBR,1-4-3-5-5,1
16,2007,2007,LUX,5-5-3-3-1,1
22,2007,2007,SVN,5-4-3-2-4,1
24,2007,2007,USA,2-4-5-5-4,1
25,2019,2019,AUS,3-3-3-5-5,1
29,2019,2019,CAN,3-3-3-5-5,1


In [9]:

# ------------------------------------------------------
# REPORT TABLES AND NOTES
# ------------------------------------------------------

report_table_profile_poset_main = profile_poset_quality_diagnostics[
    [
        "shock_id", "baseline_year", "countries", "unique_profiles",
        "dominance_relations", "hasse_edges", "layers",
        "frontier_profiles", "frontier_countries",
        "comparability_ratio", "incomparability_ratio",
    ]
].copy()

report_table_profile_poset_main["interpretation_note"] = (
    "Profile-level POSet using five final ordering variables and five ordinal levels. "
    "This is not a scalar index or total ranking."
)

report_ready_profile_poset_notes = pd.DataFrame([
    {
        "topic": "Dominance rule",
        "report_text": (
            "Profile A dominates profile B when A is at least as high as B in all five ordinal dimensions "
            "and strictly higher in at least one dimension. This produces a partial order rather than a total ranking."
        ),
    },
    {
        "topic": "Profile-level construction",
        "report_text": (
            "The main POSet is constructed at the profile level. Countries with the same five-dimensional "
            "ordinal profile occupy the same node in the Hasse diagram."
        ),
    },
    {
        "topic": "Layer interpretation",
        "report_text": (
            "Layer 1 contains undominated frontier profiles. Subsequent layers are obtained iteratively after "
            "removing the profiles in previous layers. Layer numbers should not be interpreted as a scalar resilience index."
        ),
    },
    {
        "topic": "Incomparability",
        "report_text": (
            "Incomparability is a substantive result of the POSet approach: two countries or profiles may each "
            "be stronger in different dimensions, so the method avoids forcing an arbitrary total ranking."
        ),
    },
    {
        "topic": "Digit-sum warning",
        "report_text": (
            "The profile digit-sum is used only as a layout and diagnostic aid. It is not used as a ranking rule "
            "and does not replace the dominance relation."
        ),
    },
    {
        "topic": "Temporal design",
        "report_text": (
            "The profile POSet is built separately for the 2007 and 2019 pre-shock cross-sections. The two POSet "
            "structures should therefore be interpreted as shock-specific structural snapshots."
        ),
    },
])

methodological_decision_updates_profile_poset = pd.DataFrame([
    {
        "decision": "Construct main POSet at profile level",
        "choice": "5-level ordinal profiles",
        "reason": "Profile-level construction makes the Hasse diagram interpretable and aligns with the ordinal POSet framework.",
        "risk_controlled": "Avoids presenting continuous scores as a single ranking.",
    },
    {
        "decision": "Use direct Hasse cover relations",
        "choice": "Transitive reduction of dominance relation",
        "reason": "Hasse diagrams should show only direct cover relations, not all transitive dominance relations.",
        "risk_controlled": "Improves readability and prevents visual over-cluttering.",
    },
    {
        "decision": "Keep digit-sum as layout aid only",
        "choice": "No scalar score or total ranking",
        "reason": "The project goal is structural comparison, not an Economic Resilience Index.",
        "risk_controlled": "Avoids accidental drift into scalar ranking.",
    },
])

save_table(
    report_table_profile_poset_main,
    "report_table_profile_poset_main.csv",
    "Report table profile POSet main",
    "Report-ready summary table for main profile POSet results.",
)

save_table(
    report_ready_profile_poset_notes,
    "report_ready_profile_poset_notes.csv",
    "Report-ready profile POSet notes",
    "Report-ready methodological notes for profile-level POSet construction.",
)

save_table(
    methodological_decision_updates_profile_poset,
    "methodological_decision_updates_profile_poset.csv",
    "Methodological decision updates profile POSet",
    "Decision-log entries for main profile POSet construction.",
)

display(report_table_profile_poset_main)
display(report_ready_profile_poset_notes)


Saved: report_table_profile_poset_main.csv
Saved: report_ready_profile_poset_notes.csv
Saved: methodological_decision_updates_profile_poset.csv


,shock_id,baseline_year,countries,unique_profiles,dominance_relations,hasse_edges,layers,frontier_profiles,frontier_countries,comparability_ratio,incomparability_ratio,interpretation_note
0,2007,2007,25,25,96,52,5,8,8,0.3200,0.6800,Profile-level POSet using five final ordering ...
1,2019,2019,35,34,129,64,4,12,13,0.2299,0.7701,Profile-level POSet using five final ordering ...


,topic,report_text
0,Dominance rule,Profile A dominates profile B when A is at lea...
1,Profile-level construction,The main POSet is constructed at the profile l...
2,Layer interpretation,Layer 1 contains undominated frontier profiles...
3,Incomparability,Incomparability is a substantive result of the...
4,Digit-sum warning,The profile digit-sum is used only as a layout...
5,Temporal design,The profile POSet is built separately for the ...


In [10]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "08_Profile_POSet_Main_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    profile_run_summary.to_excel(writer, sheet_name="run_summary", index=False)
    profile_poset_quality_diagnostics.to_excel(writer, sheet_name="quality_diagnostics", index=False)
    profile_nodes_with_layers.to_excel(writer, sheet_name="nodes_with_layers", index=False)
    profile_country_map_with_layers.to_excel(writer, sheet_name="country_map_layers", index=False)
    profile_hasse_edges.to_excel(writer, sheet_name="hasse_edges", index=False)
    profile_layer_summary.to_excel(writer, sheet_name="layer_summary", index=False)
    profile_pareto_profiles.to_excel(writer, sheet_name="pareto_profiles", index=False)
    profile_pareto_countries.to_excel(writer, sheet_name="pareto_countries", index=False)
    profile_incomparability_summary.to_excel(writer, sheet_name="incomparability", index=False)
    report_table_profile_poset_main.to_excel(writer, sheet_name="report_table", index=False)
    report_ready_profile_poset_notes.to_excel(writer, sheet_name="report_notes", index=False)
    methodological_decision_updates_profile_poset.to_excel(writer, sheet_name="decision_updates", index=False)
    pd.DataFrame(table_inventory_rows).to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


Audit workbook saved: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\08_Profile_POSet_Main_Audit.xlsx


In [11]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

table_inventory = pd.DataFrame(table_inventory_rows)
table_inventory.to_csv(PROCESSED_DIR / "profile_poset_main_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "profile_poset_main_table_inventory.csv", index=False)

expected_outputs = [
    "profile_country_map.csv",
    "profile_nodes.csv",
    "profile_dominance_relations.csv",
    "profile_hasse_edges.csv",
    "profile_layer_assignments.csv",
    "profile_nodes_with_layers.csv",
    "profile_country_map_with_layers.csv",
    "profile_pareto_profiles.csv",
    "profile_pareto_countries.csv",
    "profile_layer_summary.csv",
    "profile_incomparability_pairs.csv",
    "profile_incomparability_summary.csv",
    "profile_poset_quality_diagnostics.csv",
    "profile_run_summary.csv",
    "report_table_profile_poset_main.csv",
    "report_ready_profile_poset_notes.csv",
    "methodological_decision_updates_profile_poset.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("08 PROFILE POSET MAIN COMPLETE")
print("=" * 90)

display(output_check)

print("\nKey outputs to inspect/send:")
print("- 08_Profile_POSet_Main_Audit.xlsx")
print("- profile_poset_quality_diagnostics.csv")
print("- profile_layer_summary.csv")
print("- profile_pareto_countries.csv")
print("- profile_hasse_edges.csv")

print("\nNext notebook:")
print("09_Epsilon_Sensitivity_Country_Level_2002_5Var.ipynb")


08 PROFILE POSET MAIN COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,profile_country_map.csv,True,True,True
1,profile_nodes.csv,True,True,True
2,profile_dominance_relations.csv,True,True,True
3,profile_hasse_edges.csv,True,True,True
4,profile_layer_assignments.csv,True,True,True
5,profile_nodes_with_layers.csv,True,True,True
6,profile_country_map_with_layers.csv,True,True,True
7,profile_pareto_profiles.csv,True,True,True
8,profile_pareto_countries.csv,True,True,True
9,profile_layer_summary.csv,True,True,True



Key outputs to inspect/send:
- 08_Profile_POSet_Main_Audit.xlsx
- profile_poset_quality_diagnostics.csv
- profile_layer_summary.csv
- profile_pareto_countries.csv
- profile_hasse_edges.csv

Next notebook:
09_Epsilon_Sensitivity_Country_Level_2002_5Var.ipynb


In [1]:
# ======================================================
# REPORT TABLE:
# Country ordinal profile table, Fattore-style
#
# Similar to the professor example:
# Country | B1 | B2 | B3 | B4 | B5 | Country | B1 | B2 | B3 | B4 | B5
#
# For this project:
# B1 = Debt capacity
# B2 = Employment strength
# B3 = R&D intensity
# B4 = Tertiary human capital
# B5 = Energy security
#
# IMPORTANT:
# Main analysis uses 2007 and 2019.
# Do NOT use 2008 unless you intentionally want a post-shock GFC profile.
# ======================================================

from pathlib import Path
import re
import math
import numpy as np
import pandas as pd

# ------------------------------------------------------
# PATHS
# ------------------------------------------------------

PROJECT_ROOT = Path.cwd()
PROFILE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"

COUNTRY_MAP_FILE = PROFILE_DIR / "profile_country_map_with_layers.csv"

if not COUNTRY_MAP_FILE.exists():
    raise FileNotFoundError(f"Missing: {COUNTRY_MAP_FILE}")

TABLE_OUTPUT_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Profile_POSet_Main"
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_OUTPUT_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "Profile_POSet_Main"
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use 2007 and 2019 for the main report.
# If Francesco literally wants 2008, change TARGET_SHOCKS to ["2008", "2019"],
# but only if you have generated a 2008 structural-profile POSet.
TARGET_SHOCKS = ["2007", "2019"]

# ------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------

df_raw = pd.read_csv(COUNTRY_MAP_FILE)

print("Loaded:", COUNTRY_MAP_FILE)
print("Shape:", df_raw.shape)
print("Columns:")
print(df_raw.columns.tolist())

# ------------------------------------------------------
# COLUMN HELPERS
# ------------------------------------------------------

def clean_col_name(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())


def pick_col(df, candidates, df_name, required=True):
    clean_lookup = {clean_col_name(c): c for c in df.columns}

    for cand in candidates:
        key = clean_col_name(cand)
        if key in clean_lookup:
            return clean_lookup[key]

    for cand in candidates:
        cand_key = clean_col_name(cand)
        for clean_key, original_col in clean_lookup.items():
            if cand_key in clean_key or clean_key in cand_key:
                return original_col

    if required:
        raise KeyError(
            f"Could not find column in {df_name}.\n"
            f"Tried: {candidates}\n"
            f"Available: {df.columns.tolist()}"
        )

    return None


def clean_shock_id(s):
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def latex_escape(x):
    if pd.isna(x):
        return ""

    s = str(x)

    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }

    for old, new in replacements.items():
        s = s.replace(old, new)

    return s


# ------------------------------------------------------
# STANDARDIZE BASIC COLUMNS
# ------------------------------------------------------

shock_col = pick_col(
    df_raw,
    ["shock_id", "shock", "baseline_year", "year"],
    "country profile map"
)

country_col = pick_col(
    df_raw,
    ["Country", "country", "iso3", "country_code"],
    "country profile map"
)

country_name_col = pick_col(
    df_raw,
    ["country_name", "Country_Name", "name", "country_label"],
    "country profile map",
    required=False
)

profile_code_col = pick_col(
    df_raw,
    ["profile_code", "ordinal_profile", "profile", "profile_id", "profile_label"],
    "country profile map",
    required=False
)

df = df_raw.copy()
df["shock_id_std"] = clean_shock_id(df[shock_col])
df["Country_std"] = df[country_col].astype(str).str.upper().str.strip()

if country_name_col is not None:
    df["country_display"] = df[country_name_col].astype(str).str.strip()
else:
    df["country_display"] = df["Country_std"]

# ------------------------------------------------------
# IDENTIFY THE FIVE ORDINAL LEVEL COLUMNS
# ------------------------------------------------------

LEVEL_COLUMNS_PREFERRED = [
    "debt_capacity_level_5",
    "employment_strength_level_5",
    "rd_intensity_level_5",
    "human_capital_tertiary_level_5",
    "energy_security_level_5",
]

LEVEL_COLUMNS_ALT = [
    "debt_capacity_score_0_100__L5",
    "employment_strength_score_0_100__L5",
    "rd_intensity_score_0_100__L5",
    "human_capital_tertiary_score_0_100__L5",
    "energy_security_score_0_100__L5",
]

clean_to_original = {clean_col_name(c): c for c in df.columns}

def find_exact_columns(candidates):
    found = []
    for c in candidates:
        key = clean_col_name(c)
        if key in clean_to_original:
            found.append(clean_to_original[key])
    return found

level_cols = find_exact_columns(LEVEL_COLUMNS_PREFERRED)

if len(level_cols) != 5:
    level_cols = find_exact_columns(LEVEL_COLUMNS_ALT)

# If direct level columns do not exist, parse the profile code.
if len(level_cols) != 5:
    if profile_code_col is None:
        raise KeyError(
            "Could not find five ordinal level columns and could not find a profile code column to parse."
        )

    print("\nCould not find explicit level columns.")
    print(f"Trying to parse five ordinal scores from profile column: {profile_code_col}")

    def parse_profile_code(x):
        nums = re.findall(r"\d+", str(x))
        nums = [int(n) for n in nums]

        # Keep only values that look like 1--5 ordinal levels
        nums_1_5 = [n for n in nums if 1 <= n <= 5]

        if len(nums_1_5) >= 5:
            return nums_1_5[:5]

        return [np.nan] * 5

    parsed = df[profile_code_col].apply(parse_profile_code)

    for i in range(5):
        df[f"parsed_B{i+1}"] = parsed.apply(lambda x: x[i])

    level_cols = [f"parsed_B{i+1}" for i in range(5)]

print("\nUsing ordinal columns:")
for i, c in enumerate(level_cols, start=1):
    print(f"B{i}: {c}")

# ------------------------------------------------------
# CREATE STANDARD B1--B5 TABLE
# ------------------------------------------------------

b_map = {
    level_cols[0]: "B1",
    level_cols[1]: "B2",
    level_cols[2]: "B3",
    level_cols[3]: "B4",
    level_cols[4]: "B5",
}

profile_table_long = df[
    ["shock_id_std", "Country_std", "country_display"] + level_cols
].copy()

profile_table_long = profile_table_long.rename(columns=b_map)
profile_table_long = profile_table_long.rename(columns={
    "shock_id_std": "shock_id",
    "Country_std": "Country",
    "country_display": "Country name",
})

for b in ["B1", "B2", "B3", "B4", "B5"]:
    profile_table_long[b] = pd.to_numeric(profile_table_long[b], errors="coerce").astype("Int64")

# Keep one row per shock-country
profile_table_long = (
    profile_table_long
    .drop_duplicates(["shock_id", "Country"])
    .sort_values(["shock_id", "Country name"])
    .reset_index(drop=True)
)

profile_table_long.to_csv(
    TABLE_OUTPUT_DIR / "report_country_ordinal_profiles_long.csv",
    index=False
)

profile_table_long.to_csv(
    AUDIT_OUTPUT_DIR / "report_country_ordinal_profiles_long.csv",
    index=False
)

display(profile_table_long.head())

# ------------------------------------------------------
# FORMAT INTO PROFESSOR-STYLE TWO-BLOCK TABLE
# ------------------------------------------------------

def make_two_block_table(df_shock):
    df_shock = df_shock.sort_values("Country name").reset_index(drop=True)

    cols = ["Country name", "B1", "B2", "B3", "B4", "B5"]
    df_shock = df_shock[cols].copy()

    n = len(df_shock)
    split = math.ceil(n / 2)

    left = df_shock.iloc[:split].reset_index(drop=True)
    right = df_shock.iloc[split:].reset_index(drop=True)

    # Pad right side if needed
    while len(right) < len(left):
        right.loc[len(right)] = ["", pd.NA, pd.NA, pd.NA, pd.NA, pd.NA]

    two_block = pd.concat(
        [
            left.rename(columns={
                "Country name": "Country_L",
                "B1": "B1_L",
                "B2": "B2_L",
                "B3": "B3_L",
                "B4": "B4_L",
                "B5": "B5_L",
            }),
            right.rename(columns={
                "Country name": "Country_R",
                "B1": "B1_R",
                "B2": "B2_R",
                "B3": "B3_R",
                "B4": "B4_R",
                "B5": "B5_R",
            }),
        ],
        axis=1
    )

    return two_block


# ------------------------------------------------------
# LATEX EXPORT
# ------------------------------------------------------

VARIABLE_NOTE = (
    "B1 = debt capacity; "
    "B2 = employment strength; "
    "B3 = R\\&D intensity; "
    "B4 = tertiary human capital; "
    "B5 = energy security. "
    "All variables are coded on a 1--5 ordinal scale, where higher values indicate a more favourable structural condition."
)

def make_latex_table(two_block, shock_id):
    rows = []

    for _, r in two_block.iterrows():
        left_country = latex_escape(r["Country_L"])
        right_country = latex_escape(r["Country_R"])

        left_vals = []
        right_vals = []

        for b in ["B1_L", "B2_L", "B3_L", "B4_L", "B5_L"]:
            val = r[b]
            left_vals.append("" if pd.isna(val) else str(int(val)))

        for b in ["B1_R", "B2_R", "B3_R", "B4_R", "B5_R"]:
            val = r[b]
            right_vals.append("" if pd.isna(val) else str(int(val)))

        row = (
            f"{left_country} & " + " & ".join(left_vals)
            + " & "
            + f"{right_country} & " + " & ".join(right_vals)
            + r" \\"
        )

        rows.append(row)

    latex = rf"""
\begin{{table*}}[ht]
\centering
\caption{{Country structural profiles for the {shock_id} baseline}}
\label{{tab:country_profiles_{shock_id}}}
\scriptsize
\renewcommand{{\arraystretch}}{{1.12}}
\setlength{{\tabcolsep}}{{4pt}}
\begin{{tabular}}{{lccccc lccccc}}
\toprule
\textbf{{Country}} & \textbf{{B1}} & \textbf{{B2}} & \textbf{{B3}} & \textbf{{B4}} & \textbf{{B5}}
& \textbf{{Country}} & \textbf{{B1}} & \textbf{{B2}} & \textbf{{B3}} & \textbf{{B4}} & \textbf{{B5}} \\
\midrule
""" + "\n".join(rows) + rf"""
\bottomrule
\end{{tabular}}
\vspace{{2pt}}
\begin{{flushleft}}
\scriptsize \textit{{Notes:}} {VARIABLE_NOTE}
\end{{flushleft}}
\end{{table*}}
"""

    return latex


# ------------------------------------------------------
# GENERATE TABLES FOR TARGET YEARS
# ------------------------------------------------------

available_shocks = sorted(profile_table_long["shock_id"].astype(str).unique())
print("\nAvailable shock/baseline IDs:", available_shocks)

for shock_id in TARGET_SHOCKS:
    shock_id = str(shock_id)

    if shock_id not in available_shocks:
        print(f"\nWARNING: requested {shock_id}, but it is not available in the profile table.")
        print("Available IDs:", available_shocks)
        continue

    df_shock = profile_table_long[
        profile_table_long["shock_id"].astype(str) == shock_id
    ].copy()

    two_block = make_two_block_table(df_shock)

    csv_path = TABLE_OUTPUT_DIR / f"report_country_ordinal_profiles_twoblock_{shock_id}.csv"
    tex_path = TABLE_OUTPUT_DIR / f"report_country_ordinal_profiles_twoblock_{shock_id}.tex"

    two_block.to_csv(csv_path, index=False)

    latex = make_latex_table(two_block, shock_id)

    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(latex)

    print("\n" + "=" * 80)
    print(f"Created table for {shock_id}")
    print("Countries:", len(df_shock))
    print("CSV:", csv_path)
    print("LaTeX:", tex_path)
    print("=" * 80)

    print(latex)

# ------------------------------------------------------
# OPTIONAL: QUICK EXCEL AUDIT
# ------------------------------------------------------

audit_xlsx = AUDIT_OUTPUT_DIR / "report_country_ordinal_profiles_tables_audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    profile_table_long.to_excel(writer, sheet_name="long_profiles", index=False)

    for shock_id in TARGET_SHOCKS:
        shock_id = str(shock_id)

        if shock_id in available_shocks:
            df_shock = profile_table_long[
                profile_table_long["shock_id"].astype(str) == shock_id
            ].copy()
            two_block = make_two_block_table(df_shock)
            two_block.to_excel(writer, sheet_name=f"twoblock_{shock_id}", index=False)

print("\nAudit workbook saved:", audit_xlsx)

Loaded: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Profile_POSet_Main\profile_country_map_with_layers.csv
Shape: (60, 19)
Columns:
['Country', 'shock_id', 'baseline_year', 'profile_code', 'profile_digit_sum', 'debt_capacity_level_5', 'employment_strength_level_5', 'rd_intensity_level_5', 'human_capital_tertiary_level_5', 'energy_security_level_5', 'debt_capacity_score_0_100', 'employment_strength_score_0_100', 'rd_intensity_score_0_100', 'human_capital_tertiary_score_0_100', 'energy_security_score_0_100', 'main_variable_set', 'levels', 'layer', 'is_pareto_frontier']

Using ordinal columns:
B1: debt_capacity_level_5
B2: employment_strength_level_5
B3: rd_intensity_level_5
B4: human_capital_tertiary_level_5
B5: energy_security_level_5


,shock_id,Country,Country name,B1,B2,B3,B4,B5
0,2007,AUT,AUT,2,4,5,3,2
1,2007,BEL,BEL,1,2,4,4,1
2,2007,CAN,CAN,3,3,4,5,5
3,2007,CZE,CZE,4,3,3,1,5
4,2007,DEU,DEU,2,1,4,2,3



Available shock/baseline IDs: ['2007', '2019']

Created table for 2007
Countries: 25
CSV: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Outputs\Tables\Profile_POSet_Main\report_country_ordinal_profiles_twoblock_2007.csv
LaTeX: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Outputs\Tables\Profile_POSet_Main\report_country_ordinal_profiles_twoblock_2007.tex

\begin{table*}[ht]
\centering
\caption{Country structural profiles for the 2007 baseline}
\label{tab:country_profiles_2007}
\scriptsize
\renewcommand{\arraystretch}{1.12}
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lccccc lccccc}
\toprule
\textbf{Country} & \textbf{B1} & \textbf{B2} & \textbf{B3} & \textbf{B4} & \textbf{B5}
& \textbf{Country} & \textbf{B1} & \textbf{B2} & \textbf{B3} & \textbf{B4} & \textbf{B5} \\
\midrule
AUT & 2 & 4 & 5 & 3 & 2 & IRL & 4 & 4 & 2 & 4 & 1 \\
BEL & 1 & 2 &